<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/SectorLeadershipModern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --upgrade yfinance

In [24]:
import seaborn as sns
import yfinance as yf
print(yf.__version__)
import pandas as pd
#import pandas_ta as ta
import numpy as np
from datetime import datetime
import time
import matplotlib.pyplot as plt


print("Libraries Installed!")

1.4.1
Libraries Installed!


## Define Sector and Benchmark

In [44]:
tickers = [
    'XLK','XLF','XLV','XLI','XLY', 'XBI', 'GBTC',
    'XLP','XLE','XLU','XLB','XLRE','SOXX', 'GBTC', 'GLD','SLV',
    'XLC','SPY'
]

prices = yf.download(
    tickers,
    start="2020-01-01",
    auto_adjust=True
)['Close']

weekly = prices.resample('W-FRI').last()

current_week_end = weekly.index[-1]
previous_week_end = weekly.index[-2]

# =============================
# 2. SCORE FUNCTION (FIXED RS + ACCEL)
# =============================
def calculate_score(data, as_of):

    df = data.loc[:as_of]

    sector = df.drop(columns="SPY")

    # -----------------------------
    # Relative Strength
    # -----------------------------
    rs = sector.div(df["SPY"], axis=0)

    # -----------------------------
    # RS Momentum (4-week change)
    # -----------------------------
    rs_momentum = rs.pct_change(4)

    # -----------------------------
    # RS Acceleration (change in momentum)
    # -----------------------------
    rs_accel = rs_momentum.diff(1)

    # -----------------------------
    # RS Slope (trend of RS)
    # -----------------------------
    rs_slope = (
        rs.rolling(4).mean().iloc[-1] -
        rs.rolling(4).mean().iloc[-4]
    )

    # -----------------------------
    # Latest snapshot
    # -----------------------------
    score = pd.DataFrame({
        "RS_Strength": rs.iloc[-1],
        "RS_Momentum": rs_momentum.iloc[-1],
        "RS_Accel": rs_accel.iloc[-1],
        "RS_Slope": rs_slope
    })

    # Composite rank score
    score["Total"] = score.rank(ascending=False).mean(axis=1)

    return score


# =============================
# 3. CURRENT / PREVIOUS SCORES
# =============================
current_score = calculate_score(weekly, current_week_end)
previous_score = calculate_score(weekly, previous_week_end)

# =============================
# 4. RANKS
# =============================
curr_rank = current_score["Total"].rank(ascending=False, method="first")
prev_rank = previous_score["Total"].rank(ascending=False, method="first")

# =============================
# 5. IMPACT (NOW USING ACCELERATION)
# =============================
impact = (prev_rank - curr_rank) * current_score["RS_Accel"].abs()

# =============================
# 6. ROTATION TABLE
# =============================
df = pd.DataFrame({
    "Curr Rank": curr_rank,
    "Prev Rank": prev_rank,
    "Rank Change": prev_rank - curr_rank,
    "RS_Strength": current_score["RS_Strength"],
    "RS_Momentum": current_score["RS_Momentum"],
    "RS_Accel": current_score["RS_Accel"],
    "Impact": impact
})

df = df.sort_values("Impact", ascending=False)

# =============================
# 7. TOP 5 ROTATION RATE
# =============================
N = 5

top_now = set(current_score.sort_values("Total").head(N).index)
top_prev = set(previous_score.sort_values("Total").head(N).index)

rotating_in = top_now - top_prev
rotating_out = top_prev - top_now

rotation_rate = len(rotating_in) / N

# =============================
# 8. REGIME CLASSIFICATION
# =============================
if rotation_rate == 0:
    regime = "Stable Leadership"
elif rotation_rate <= 0.2:
    regime = "Low Rotation"
elif rotation_rate <= 0.4:
    regime = "Mild Rotation"
elif rotation_rate <= 0.6:
    regime = "Moderate Rotation"
else:
    regime = "High Rotation"

# =============================
# 9. FLOW LABELS
# =============================
def flow(x):
    if x >= 3:
        return "Strong Inflow 🚀"
    elif x >= 1:
        return "Mild Inflow 📈"
    elif x <= -3:
        return "Strong Outflow 🔻"
    elif x <= -1:
        return "Mild Outflow 📉"
    else:
        return "Neutral"

df["Flow"] = df["Rank Change"].apply(flow)

# =============================
# 10. OUTPUT
# =============================
print("\n==============================")
print("SECTOR ROTATION ENGINE (v2)")
print("==============================")

print(f"Current Week  : {current_week_end.date()}")
print(f"Previous Week : {previous_week_end.date()}")
print(f"Rotation Rate : {rotation_rate:.0%}")
print(f"Regime        : {regime}")

print("\nRotating IN:")
print(rotating_in if rotating_in else "None")

print("\nRotating OUT:")
print(rotating_out if rotating_out else "None")

print("\n==============================")
print("IMPACT TABLE")
print("==============================")

df

[*********************100%***********************]  17 of 17 completed


SECTOR ROTATION ENGINE (v2)
Current Week  : 2026-06-26
Previous Week : 2026-06-19
Rotation Rate : 20%
Regime        : Low Rotation

Rotating IN:
{'XLF'}

Rotating OUT:
{'XLK'}

IMPACT TABLE


,Curr Rank,Prev Rank,Rank Change,RS_Strength,RS_Momentum,RS_Accel,Impact,Flow
Ticker,,,,,,,,
XLK,8.0,15.0,7.0,0.251538,-0.005231,-0.062201,0.435406,Strong Inflow 🚀
SOXX,13.0,16.0,3.0,0.848792,0.125938,-0.059869,0.179608,Strong Inflow 🚀
GLD,3.0,9.0,6.0,0.504445,-0.087500,-0.019200,0.115202,Strong Inflow 🚀
SLV,1.0,2.0,1.0,0.071735,-0.207866,-0.074888,0.074888,Mild Inflow 📈
XLB,6.0,10.0,4.0,0.070477,0.043528,0.017464,0.069856,Strong Inflow 🚀
XLY,5.0,8.0,3.0,0.154689,-0.032424,-0.011505,0.034514,Strong Inflow 🚀
GBTC,2.0,4.0,2.0,0.062772,-0.169348,0.004775,0.009550,Mild Inflow 📈
XLC,4.0,7.0,3.0,0.144265,-0.056646,-0.000764,0.002293,Strong Inflow 🚀
XLI,14.0,13.0,-1.0,0.250059,0.092492,0.043535,-0.043535,Mild Outflow 📉


In [46]:
# =============================
# SECTOR SIGNAL CLASSIFICATION
# =============================

def sector_signal(row, rank_series):

    rank = rank_series[row.name]
    accel = row["RS_Accel"]
    rank_change = row["Rank Change"]

    # Top, mid, bottom segmentation
    if rank <= 5:
        tier = "top"
    elif rank <= 10:
        tier = "mid"
    else:
        tier = "weak"

    # -----------------------------
    # BUY CONDITIONS
    # -----------------------------
    if tier == "top" and accel > 0 and rank_change > 0:
        return "BUY 🟢"

    if tier == "top" and accel > 0:
        return "BUY 🟢 (early)"

    # -----------------------------
    # WATCH CONDITIONS
    # -----------------------------
    if tier == "mid" and accel >= 0:
        return "WATCH 🟡 (improving)"

    if tier == "top" and accel <= 0:
        return "WATCH 🟡 (late cycle)"

    if tier == "mid" and rank_change > 0:
        return "WATCH 🟡 (building)"

    # -----------------------------
    # AVOID CONDITIONS
    # -----------------------------
    if tier == "weak" and accel < 0:
        return "AVOID 🔴"

    if rank_change < 0 and accel < 0:
        return "AVOID 🔴 (distribution)"

    return "WATCH 🟡"

In [47]:
df["Signal"] = df.apply(
    sector_signal,
    axis=1,
    rank_series=curr_rank
)

final_view = df.sort_values(
    ["Signal", "Impact"],
    ascending=[True, False]
)

print("\n==============================")
print("SECTOR SIGNAL DASHBOARD")
print("==============================")

final_view


SECTOR SIGNAL DASHBOARD


,Curr Rank,Prev Rank,Rank Change,RS_Strength,RS_Momentum,RS_Accel,Impact,Flow,Signal
Ticker,,,,,,,,,
SOXX,13.0,16.0,3.0,0.848792,0.125938,-0.059869,0.179608,Strong Inflow 🚀,AVOID 🔴
GBTC,2.0,4.0,2.0,0.062772,-0.169348,0.004775,0.009550,Mild Inflow 📈,BUY 🟢
XLI,14.0,13.0,-1.0,0.250059,0.092492,0.043535,-0.043535,Mild Outflow 📉,WATCH 🟡
XLF,12.0,11.0,-1.0,0.073073,0.072688,0.045471,-0.045471,Mild Outflow 📉,WATCH 🟡
XBI,16.0,14.0,-2.0,0.205954,0.137995,0.073498,-0.146996,Mild Outflow 📉,WATCH 🟡
XLV,15.0,12.0,-3.0,0.212447,0.077178,0.084472,-0.253417,Strong Outflow 🔻,WATCH 🟡
XLP,11.0,6.0,-5.0,0.114426,0.048577,0.070233,-0.351166,Strong Outflow 🔻,WATCH 🟡
XLK,8.0,15.0,7.0,0.251538,-0.005231,-0.062201,0.435406,Strong Inflow 🚀,WATCH 🟡 (building)
XLB,6.0,10.0,4.0,0.070477,0.043528,0.017464,0.069856,Strong Inflow 🚀,WATCH 🟡 (improving)
